# Exploring EW4 data - GPM IMERG
This notebook explore the GPM IMERG dataset, including loading in PyEarthTools.

This dataset is a subset of the [NASA Integrated Multi-satellite Retrieval (IMERG)](https://gpm.nasa.gov/data/imerg) data for Global Precipitation Measurement (GPM). This subset has the following extents:
- Temporal Extent: April to September 2025
- Spatial Extent
  - Latitude 0N to 20N
  - Longitude 27W to 20E


### Import libraries

In [1]:
import pathlib
import datetime

In [2]:
import xarray

In [3]:
import matplotlib
import cartopy.crs

In [4]:
import site_archive_jasmin

ROOT_DIRECTORIES: {'ERA5lowres': '/gws/nopw/j04/mohc_shared/dscop/weatherbench/5.625deg/', 'MOGLOBAL': '/gws/nopw/j04/mohc_shared//dscop/mo_pet_site_archive/mo_global/', 'MOUKV': '/gws/nopw/j04/mohc_shared//dscop/mo_pet_site_archive/mo_ukv/', 'Himawari': '/gws/ssde/j25a/mmh_storage/theme3/rv74_himawari', 'HimawariChannels': '/gws/ssde/j25a/mmh_storage/theme3/ra22_himawari', 'Rainfields3': '/gws/ssde/j25a/mmh_storage/theme3/rq0Radar', 'ew4_imerg_precip': '/gws/nopw/j04/ew4energy/imerg_2025_summer'}


In [5]:
import pyearthtools

In [6]:
import pyearthtools.data as petdata
import pyearthtools.pipeline as petpipe

In [7]:
from pyearthtools.data import Petdt
from pyearthtools.data.exceptions import DataNotFoundError
from pyearthtools.data.indexes import ArchiveIndex, decorators
from pyearthtools.data.transforms import Transform, TransformCollection
from pyearthtools.data.archive import register_archive


In [ ]:
from site_archive_jasmin.utilities import (
    cached_exists,
    cached_iterdir,
)  # Could these be moved into a generic module?


## Explore the data in the dataset
Lets start by looking at a sample data file, before demonstrating loading the data in pyearthtools

In [ ]:
ew4_gws_dir = pathlib.Path('/gws/nopw/j04/ew4energy/')
imerg_ew4_dir = ew4_gws_dir / 'imerg_2025_summer'


In [ ]:
def get_imerg_path(start_dt, time_delta, fname_template, data_dir):
    day_minutes = start_dt.hour * 60 + start_dt.minute
    date_str = '{dt.year:04d}{dt.month:02d}{dt.day:02d}'.format(dt=start_dt)
    time_template = '{dt.hour:02d}{dt.minute:02d}{dt.second:02d}'
    start_time = time_template.format(dt=start_dt)
    end_time = time_template.format(dt=start_dt+time_delta-datetime.timedelta(seconds=1))
    imerg_path = data_dir / fname_template.format(date_str=date_str,
                                              start_time=start_time,
                                              end_time=end_time,
                                                  day_minutes=day_minutes,
                                             )
    return imerg_path


In [ ]:
imerg_fname_template = '3B-HHR-E.MS.MRG.3IMERG.{date_str}-S{start_time}-E{end_time}.{day_minutes:04d}.V07B.HDF5.SUB.nc4'

In [ ]:
start_dt = datetime.datetime(2025,5,1,0,0)
end_dt = datetime.datetime(2025,6,1,0,0)
time_delta=datetime.timedelta(minutes=30)

In [ ]:
num_files = int((end_dt - start_dt) / time_delta)

In [ ]:
num_files

In [ ]:
imerg_filelist = [
    get_imerg_path(start_dt + (time_delta * time_ix),
                   time_delta,
                   imerg_fname_template,
                   imerg_ew4_dir
                  )
    for time_ix in range(num_files) ]


In [ ]:
ew4_imerge_sample_ds = xarray.open_mfdataset(imerg_filelist)

In [ ]:
imerg_filelist[0]

In [ ]:
ew4_imerge_sample_ds.to_netcdf('/gws/nopw/j04/ew4energy/imerg_pet_tutorial/imerg_late_202505.nc')

In [ ]:
ew4_imerge_sample_ds

In [ ]:
ew4_imerge_sample_ds['precipitation']


In [ ]:
float(min(ew4_imerge_sample_ds['lat'])), float(max(ew4_imerge_sample_ds['lat'])), float(min(ew4_imerge_sample_ds['lon'])), float(max(ew4_imerge_sample_ds['lon']))

In [ ]:
precip_plot_bins = [1e-3,0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 4.0, 8.0, 16.0, 32.0]

In [ ]:
# ew4_imerge_sample_ds['precipitation'].plot.hist(bins=precip_plot_bins)


In [ ]:
# fig1 = matplotlib.pyplot.figure(figsize=(16,10))
# ax1=fig1.add_subplot(1,1,1,projection=cartopy.crs.PlateCarree())
# ew4_imerge_sample_ds['precipitation'][100].plot.contourf(ax=ax1)
# ax1.coastlines(color='w')

## Loading the Data in PyEarthTools
Now we will load the data through the PyEarthTools data accessor. This is a class that has been created which how to access the data. For a dataset like this which is stored as files on disk, it specifries which files to load. Data could be loaded through other mechanisms though, for example it could be load from a database, through a web api, from a tape archive,  from a cloud-based object store or many other mechanisms.

In [ ]:
import importlib


In [ ]:
site_archive_jasmin.MOGLOBAL(["2m_temperature", "u", "v"])['2018-09-12 18:00']

In [ ]:
era5_acc = site_archive_jasmin.ERA5lowres(["2m_temperature", "u", "v"])

In [ ]:
era5_acc['2005']

In [8]:
ew4_imerg_accessor = site_archive_jasmin.Ew4Imerg('2025-04-01 00:00', '2025-07-01 00:00')

2025-04-01 00:00
2025-07-01 00:00


In [9]:
ew4_imerg_accessor

Ew4Imerg
	Description                    GPM IMERG - EW4Energy project 
		 range                          '2025-04-01 to 2025-09-30'
		 Documentation                  'https://gpm.nasa.gov/data/imerg'


	Initialisation                 
		 end                            '2025-07-01 00:00'
		 start                          '2025-04-01 00:00'
	Transforms                     
		 StandardCoordinateNames        {'latitude': "['lat', 'Latitude', 'yt_ocean', 'yt']", 'longitude': "['lon', 'Longitude', 'xt_ocean', 'xt']", 'replacement_dictionary': 'None', 'time': "['Time']"}

In [13]:
ew4_imerg_accessor['2025-05-23']



2025-05-23T00:00
> /home/users/shaddad/prog/pyearthtools_jasmin/src/site_archive_jasmin/ew4_imerg.py(129)filesystem()
    126         import pdb
    127         pdb.set_trace()
    128 
--> 129         return paths
    130 



ipdb>  c


2025-05-23T00:00
> /home/users/shaddad/prog/pyearthtools_jasmin/src/site_archive_jasmin/ew4_imerg.py(129)filesystem()
    126         import pdb
    127         pdb.set_trace()
    128 
--> 129         return paths
    130 



ipdb>  c


2025-05-23T00:30
> /home/users/shaddad/prog/pyearthtools_jasmin/src/site_archive_jasmin/ew4_imerg.py(129)filesystem()
    126         import pdb
    127         pdb.set_trace()
    128 
--> 129         return paths
    130 



...

2025-05-23T23:30
> /home/users/shaddad/prog/pyearthtools_jasmin/src/site_archive_jasmin/ew4_imerg.py(129)filesystem()
    126         import pdb
    127         pdb.set_trace()
    128 
--> 129         return paths
    130 



ipdb>  cc


*** NameError: name 'cc' is not defined


ipdb>  c


<xarray.Dataset> Size: 18MB
Dimensions:        (time: 48, longitude: 469, latitude: 202)
Coordinates:
  * time           (time) datetime64[ns] 384B 2025-05-23 ... 2025-05-23T23:30:00
  * longitude      (longitude) float32 2kB -27.05 -26.95 -26.85 ... 19.65 19.75
  * latitude       (latitude) float32 808B 0.05 0.15 0.25 ... 19.95 20.05 20.15
Data variables:
    precipitation  (time, longitude, latitude) float32 18MB dask.array<chunksize=(1, 469, 202), meta=np.ndarray>
Attributes:
    CDI:                                    Climate Data Interface version 1....
    Conventions:                            CF-1.6
    Original_Producer_Metadata_FileInfo:    DataFormatVersion=7e;\nTKCodeBuil...
    Original_Producer_Metadata_GridHeader:  BinMethod=ARITHMETIC_MEAN;\nRegis...
    CDO:                                    Climate Data Operators version 1....

In [11]:
ew4_imerg_accessor['2025-05-23 03:00']

2025-05-23T03:00
> /home/users/shaddad/prog/pyearthtools_jasmin/src/site_archive_jasmin/ew4_imerg.py(129)filesystem()
    126         import pdb
    127         pdb.set_trace()
    128 
--> 129         return paths
    130 



ipdb>  paths


{'precipitation': PosixPath('/gws/nopw/j04/ew4energy/imerg_2025_summer/3B-HHR-E.MS.MRG.3IMERG.20250523-S030000-E032959.0180.V07B.HDF5.SUB.nc4')}


ipdb>  c


DataNotFoundError: Data with args: (Petdt('2025-05-23T03:00'),) could not be found.

In [12]:
pathlib.Path('/gws/nopw/j04/ew4energy/imerg_2025_summer/3B-HHR-E.MS.MRG.3IMERG.20250523-S030000-E032959.0180.V07B.HDF5.SUB.nc4').is_file()

True

In [ ]:
xarray.open_dataset('/gws/nopw/j04/ew4energy/imerg_2025_summer/3B-HHR-E.MS.MRG.3IMERG.20250523-S030000-E032959.0180.V07B.HDF5.SUB.nc4')

In [ ]:
ghana_extents = {
    'latitude': (4.5,11.25),
    'longitude': (-3.5, 1.5),
}


In [ ]:
ghana_pet_box = (ghana_extents['latitude'][0],
                 ghana_extents['longitude'][0],
                 ghana_extents['latitude'][1],
                 ghana_extents['longitude'][1],
                )

In [ ]:
ew4_imerg_pipeline = petpipe.Pipeline(
    ew4_imerg_accessor,
    petpipe.modifications.TemporalRetrieval(
        concat=True,
        samples=((0,3,1))),
    petdata.transform.region.Bounding(*ghana_pet_box),  
    iterator=petpipe.iterators.DateRange('20250501T00', '20250511T00', interval='3 hours'), 
    exceptions_to_ignore=petdata.exceptions.DataNotFoundError,
)


In [ ]:
ew4_imerg_pipeline['2025-05-05']

In [ ]:
ew4_imerg_iter = iter(ew4_imerg_pipeline)

In [ ]:
next(ew4_imerg_iter), 

In [ ]:
next(ew4_imerg_iter), 

In [ ]:
next(ew4_imerg_iter), next(ew4_imerg_iter),

In [ ]:
# load a sample file to find contents, especially variables

In [ ]:
ew4_imerg_tw_pipe = petpipe.Pipeline(
    ew4_imerg_accessor,
    petpipe.modifications.TemporalWindow(prior_indexes=[-2,-1], posterior_indexes=[0], timedelta='30 minutes'),
    petdata.transform.region.Bounding(*ghana_pet_box),  
    iterator=petpipe.iterators.DateRange('20250501T00', '20250511T00', interval='3 hours'), 
    exceptions_to_ignore=petdata.exceptions.DataNotFoundError,
)


In [ ]:
#todo
# - find start and end time for data and select 1 day
# get all files
# load into dataset
# create a PET accessor to load those files
# demonstrate visualising
# create a demo autoencoder for the data
